In [1]:
from datasets import load_dataset, load_from_disk
from replay.metrics import HitRate
import polars as pl
import pandas as pd
import json
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from torch.utils.tensorboard import SummaryWriter
import faiss
from functools import reduce
import datasets
import torch
import torch.nn as nn
from tqdm import tqdm
import os
from datetime import datetime

pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(-1)

DATA_PATH = "/home/jupyter/filestore/storage/datasets"

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2026-04-29 22:57:03.571094: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-29 22:57:04.796977: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following ins

In [2]:
dataset = load_from_disk(f"{DATA_PATH}/user_carts_20230501")
polars_ds = dataset.to_polars()

In [6]:
polars_ds.shape

(2895617, 5)

In [7]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")
TEST_END_DT = pd.to_datetime("2023-05-21")

train_cart_adds = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TRAIN_END_DT)
)

train_cart_adds.shape

(1393010, 6)

In [9]:
sessions_with_cart_adds = (
    train_cart_adds
    .groupby("session_id", "user_id")
    .agg(pl.n_unique("item_id").alias("cart_adds_cnt"))
    .filter(pl.col("cart_adds_cnt") >= 2)
)

sessions_with_cart_adds.shape

(209965, 3)

In [10]:
session_cart_adds = (
    train_cart_adds
    .join(
        sessions_with_cart_adds,
        on=["session_id", "user_id"],
        how="inner"
    )
    .select(
        pl.struct("session_id", "user_id").apply(lambda x: f"{x['session_id']}_{x['user_id']}"),
        pl.col("item_id")
    )
    .unique()
)

In [11]:
from collections import defaultdict, Counter

item_occurances = defaultdict(list)

session_with_item = session_cart_adds.groupby("item_id").agg(pl.col("session_id"))
session_items = session_cart_adds.groupby("session_id").agg(pl.col("item_id"))
session_items = dict(zip(session_items["session_id"].to_list(), session_items["item_id"].to_list()))

In [12]:
item_ids = session_with_item["item_id"].to_list()
session_ids = session_with_item["session_id"].to_list()

recs_lens = []

for item_id, item_sessions in zip(item_ids, session_ids):
    for session in item_sessions:
        co_items = session_items[session]
        for item in co_items:
            if item != item_id:
                item_occurances[item_id].append(item)
    item_occurances[item_id] = list(set(item_occurances[item_id]))
    recs_lens.append(len(item_occurances[item_id]))
    
recs_lens = np.array(recs_lens)

In [13]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")
TEST_END_DT = pd.to_datetime("2023-05-21")

train_interactions = (
    load_from_disk("/home/jupyter/filestore/storage/datasets/user_events_20230501")
    .to_polars()
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TRAIN_END_DT)
)

all_clicks = (
    train_interactions
    .select(
        pl.col("user_id"),
        pl.col("c2_name"),
        pl.col("name"),
        pl.col("item_id"),
        pl.col("stime"),
        pl.col("stime").rank("dense", descending=True).over("user_id").alias("rn")
    )
)

In [35]:
user_last_clicks_20 = (
     all_clicks
    .filter(pl.col("rn") <= 200)
    .groupby("user_id")
    .agg(pl.col("item_id").alias("last_clicks"))
)

In [15]:
def get_similar_items(row):
    return list(reduce(lambda x, y: x + y, [item_occurances[item_id] for item_id in row["last_clicks"] if item_id in item_occurances], []))

In [36]:
recs_20 = (
    user_last_clicks_20
    .with_columns(
        pl.struct(["last_clicks"]).apply(get_similar_items).alias("recs")
    )
)

In [40]:
recs_20.rename({"recs": "co_cart_recs"}).select("user_id", "co_cart_recs").write_parquet("co_cart_recs.parquet")

In [31]:
test_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") > TRAIN_END_DT)
    .filter(pl.col("date") <=  TEST_END_DT)
    .groupby("user_id")
    .agg(pl.col("item_id").alias("future_clicks"))
    .join(
        recs_20,
        on="user_id",
        how="inner"
    )
    .with_columns(pl.col("recs").apply(len).alias("recs_count"))
)

In [32]:
recs_stats = (
    test_interactions
    .select(
        pl.min("recs_count").alias("min_recs_count"),
        pl.mean("recs_count").alias("mean_recs_count"),
        pl.max("recs_count").alias("max_recs_count"),
    )
)

recs_stats.head(1)

min_recs_count,mean_recs_count,max_recs_count
i64,f64,i64
0,41.971672,3926


In [19]:
from replay.metrics import Recall, Precision, HitRate

In [33]:
TOP_K_VALUES = [10, 100, 1000]

def calc_recall(row):
    return Recall._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_precision(row):
    return Precision._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_hitrate(row):
    return HitRate._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def intersection(row):
    return len(set(row["recs"]) & set(row["future_clicks"]))

metrics = (
    test_interactions
    .with_columns(
        pl.struct(["future_clicks", "recs"]).apply(calc_recall).alias("recall"),
        pl.struct(["future_clicks", "recs"]).apply(calc_precision).alias("precision"),
        pl.struct(["future_clicks", "recs"]).apply(calc_hitrate).alias("hitrate"),
    )
    .select(
        pl.col("recall").arr.get(0).mean().alias("recall@10"),
        pl.col("recall").arr.get(1).mean().alias("recall@100"),
        pl.col("recall").arr.get(2).mean().alias("recall@1000"),
        pl.col("precision").arr.get(0).mean().alias("precision@10"),
        pl.col("precision").arr.get(1).mean().alias("precision@100"),
        pl.col("precision").arr.get(2).mean().alias("precision@1000"),
        pl.col("hitrate").arr.get(0).mean().alias("hitrate@10"),
        pl.col("hitrate").arr.get(1).mean().alias("hitrate@100"),
        pl.col("hitrate").arr.get(2).mean().alias("hitrate@1000"),
        pl.col("hitrate").arr.get(0).sum().alias("hitrate_sum@10"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(1).sum().alias("hitrate_sum@100"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(2).sum().alias("hitrate_sum@1000") # количество рекомендаций, попавших в отложенную выборку
    )
    .head(5)
)

In [34]:
metrics # 300 событий

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.007137,0.014187,0.015465,0.002072,0.000499,0.00006,0.016203,0.034363,0.038984,298.0,632.0,717.0


In [28]:
metrics # 200 событий

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.007596,0.014085,0.015102,0.002213,0.000489,0.000057,0.017181,0.034472,0.037734,316.0,634.0,694.0


In [ ]:
metrics # 100 событий

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.007952,0.013332,0.013725,0.002316,0.000465,0.00005,0.017779,0.03246,0.033602,327.0,597.0,618.0
